# Artificial Neural Network (ANN) for Customer Churn Prediction

---

## Problem Statement

### Business Context
A telecommunications company wants to predict which customers are likely to churn (leave the service) based on their usage patterns and demographic information. This will help the company take proactive measures to retain valuable customers.

### Dataset Features
- **Customer Demographics**: age, gender, tenure
- **Service Usage**: monthly charges, total charges, contract type
- **Services Subscribed**: internet service, online security, tech support
- **Target Variable**: Churn (Yes/No)

### Objective
Build an Artificial Neural Network to predict customer churn
---

## Step 1: Import Required Libraries

We'll import all necessary libraries for data manipulation and deep learning.

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score

# Deep Learning libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


## Step 2: Load and Explore the Dataset

For this demonstration, we'll create a synthetic dataset that mimics real-world customer churn data.

**Note**: In a real project, you would load data using `pd.read_csv('your_data.csv')`

In [ ]:
def create_customer_data(n_samples=2000):
    """
    Create synthetic customer churn dataset

    Parameters:
    -----------
    n_samples : int
        Number of samples to generate

    Returns:
    --------
    DataFrame with customer information
    """
    np.random.seed(42)

    data = {
        'tenure': np.random.randint(1, 72, n_samples),
        'MonthlyCharges': np.random.uniform(20, 120, n_samples),
        'TotalCharges': np.random.uniform(20, 8000, n_samples),
        'gender': np.random.choice(['Male', 'Female'], n_samples),
        'SeniorCitizen': np.random.choice([0, 1], n_samples, p=[0.85, 0.15]),
        'Partner': np.random.choice(['Yes', 'No'], n_samples),
        'Dependents': np.random.choice(['Yes', 'No'], n_samples),
        'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
        'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.4, 0.4, 0.2]),
        'OnlineSecurity': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'TechSupport': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples)
    }

    df = pd.DataFrame(data)

    # Create realistic churn probability based on multiple factors
    churn_prob = (
        (df['tenure'] < 12).astype(int) * 0.35 +
        (df['MonthlyCharges'] > 80).astype(int) * 0.25 +
        (df['Contract'] == 'Month-to-month').astype(int) * 0.30 +
        (df['SeniorCitizen'] == 1).astype(int) * 0.15 +
        (df['OnlineSecurity'] == 'No').astype(int) * 0.10 +
        (df['TechSupport'] == 'No').astype(int) * 0.10
    ) / 1.25

    churn_prob = np.clip(churn_prob + np.random.normal(0, 0.1, n_samples), 0, 1)
    df['Churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

    return df

# Create the dataset
print("Creating customer dataset...")
df = create_customer_data(2000)

print("\nDataset created successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"Number of samples: {df.shape[0]}")

Creating customer dataset...

Dataset created successfully!
Dataset shape: (2000, 13)
Number of features: 12
Number of samples: 2000


### Display Sample Data

In [ ]:
# Display first few rows
print("\nFirst 5 rows of the dataset:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())


First 5 rows of the dataset:
   tenure  MonthlyCharges  TotalCharges  gender  SeniorCitizen Partner  \
0      52       45.467062   5263.103376  Female              1      No   
1      15      104.087158   7189.124003    Male              0      No   
2      61       23.842635   3208.159901  Female              0      No   
3      21      110.176199   2627.821753    Male              0     Yes   
4      24       66.147746    106.272447  Female              0      No   

  Dependents        Contract InternetService       OnlineSecurity TechSupport  \
0        Yes        One year     Fiber optic  No internet service         Yes   
1         No  Month-to-month     Fiber optic                  Yes          No   
2        Yes        Two year     Fiber optic                   No         Yes   
3         No        One year              No  No internet service         Yes   
4        Yes        Two year             DSL  No internet service         Yes   

      PaymentMethod  Churn  
0       C

### Check Target Variable Distribution

In [ ]:
# Check class distribution
print("\nTarget Variable (Churn) Distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn Rate: {df['Churn'].mean() * 100:.2f}%")
print(f"Not Churned: {(df['Churn']==0).sum()} ({(df['Churn']==0).sum()/len(df)*100:.2f}%)")
print(f"Churned: {(df['Churn']==1).sum()} ({(df['Churn']==1).sum()/len(df)*100:.2f}%)")


Target Variable (Churn) Distribution:
Churn
0    1387
1     613
Name: count, dtype: int64

Churn Rate: 30.65%
Not Churned: 1387 (69.35%)
Churned: 613 (30.65%)


## Step 3: Data Preprocessing

### 3.1 Handle Categorical Variables

We'll encode categorical variables into numerical format using Label Encoding.

In [ ]:

# Create a copy of the dataframe
df_processed = df.copy()

# Identify categorical columns
categorical_columns = df_processed.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {categorical_columns}")

# Label Encoding for categorical variables
label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"✓ Encoded {col}: {dict(enumerate(le.classes_))}")

print("\nCategorical encoding completed!")
print("\nProcessed data sample:")
print(df_processed.head())

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'Contract', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaymentMethod']
✓ Encoded gender: {0: 'Female', 1: 'Male'}
✓ Encoded Partner: {0: 'No', 1: 'Yes'}
✓ Encoded Dependents: {0: 'No', 1: 'Yes'}
✓ Encoded Contract: {0: 'Month-to-month', 1: 'One year', 2: 'Two year'}
✓ Encoded InternetService: {0: 'DSL', 1: 'Fiber optic', 2: 'No'}
✓ Encoded OnlineSecurity: {0: 'No', 1: 'No internet service', 2: 'Yes'}
✓ Encoded TechSupport: {0: 'No', 1: 'No internet service', 2: 'Yes'}
✓ Encoded PaymentMethod: {0: 'Bank transfer', 1: 'Credit card', 2: 'Electronic check', 3: 'Mailed check'}

Categorical encoding completed!

Processed data sample:
   tenure  MonthlyCharges  TotalCharges  gender  SeniorCitizen  Partner  \
0      52       45.467062   5263.103376       0              1        0   
1      15      104.087158   7189.124003       1              0        0   
2      61       23.842635   3208.159901       0              0

### 3.2 Split Features and Target

In [ ]:
# Separate features (X) and target (y)
X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

Features shape: (2000, 12)
Target shape: (2000,)

Feature columns: ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaymentMethod']


### 3.3 Train-Test Split

We'll split the data into training (80%) and testing (20%) sets.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data Split Summary:")
print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set churn rate: {y_train.mean()*100:.2f}%")
print(f"Test set churn rate: {y_test.mean()*100:.2f}%")

Data Split Summary:
Training set size: 1600 samples (80.0%)
Test set size: 400 samples (20.0%)

Training set churn rate: 30.63%
Test set churn rate: 30.75%


### 3.4 Feature Scaling

Neural networks perform better when features are scaled. We'll use StandardScaler to normalize the features.

In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")

# Show scaling statistics
print("\nScaling Statistics (Training Set):")
print(f"Mean (first 5 features): {X_train_scaled.mean(axis=0)[:5]}")
print(f"Std (first 5 features): {X_train_scaled.std(axis=0)[:5]}")

Feature scaling completed!

Scaled training data shape: (1600, 12)
Scaled test data shape: (400, 12)

Scaling Statistics (Training Set):
Mean (first 5 features): [-1.50990331e-16  2.86437540e-16  1.52655666e-17 -3.44169138e-17
  2.66453526e-17]
Std (first 5 features): [1. 1. 1. 1. 1.]


## Step 4: Build the ANN Model

### Architecture Overview:
- **Input Layer**: Number of features
- **Hidden Layer 1**: 64 neurons with ReLU activation + Dropout (30%)
- **Hidden Layer 2**: 32 neurons with ReLU activation + Dropout (30%)
- **Hidden Layer 3**: 16 neurons with ReLU activation + Dropout (20%)
- **Output Layer**: 1 neuron with Sigmoid activation (binary classification)

### Key Components:
- **Dropout**: Prevents overfitting by randomly dropping neurons during training
- **ReLU Activation**: Introduces non-linearity, helps model learn complex patterns
- **Sigmoid Activation**: Outputs probability between 0 and 1 for binary classification

In [ ]:
print(" Building ANN Model...\n")

# Initialize Sequential model
## Sequential Model:  Layer by layer data passses through each of the layers present
model = Sequential(name='Customer_Churn_ANN')
## Layers: Layers are basically the workstations where data is processed and transformmed basis the need.
## Input -> Layers (Process) -> Output

## ReLU: f(x) = max(0,x)

# Input Layer + First Hidden Layer
model.add(Dense(
    units=64,
    activation='relu',
    input_shape=(X_train_scaled.shape[1],),
    name='Hidden_Layer_1'
))
model.add(Dropout(0.3, name='Dropout_1'))

# Second Hidden Layer
model.add(Dense(
    units=32,
    activation='relu',
    name='Hidden_Layer_2'
))
model.add(Dropout(0.3, name='Dropout_2'))

# Third Hidden Layer
model.add(Dense(
    units=16,
    activation='relu',
    name='Hidden_Layer_3'
))
model.add(Dropout(0.2, name='Dropout_3'))

# Output Layer
model.add(Dense(
    units=1,
    activation='sigmoid',
    name='Output_Layer'
))



 Building ANN Model...



### Model Summary

Let's view the model architecture and see the number of parameters.

## Step 5: Compile the Model

We need to configure the model with:
- **Optimizer**: Adam (adaptive learning rate)
- **Loss Function**: Binary Crossentropy (for binary classification)
- **Metrics**: Accuracy

In [ ]:
print("Compiling the model...\n")

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")
print("\nModel Configuration:")
print(f"Optimizer: Adam")
print(f"Loss Function: Binary Crossentropy")
print(f"Metrics: Accuracy")

Compiling the model...

Model compiled successfully!

Model Configuration:
Optimizer: Adam
Loss Function: Binary Crossentropy
Metrics: Accuracy


## Step 6: Setup Callbacks

Callbacks help improve training:
- **EarlyStopping**: Stops training when validation performance stops improving
- **ModelCheckpoint**: Saves the best model during training

In [ ]:
# Early Stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# Model Checkpoint callback
checkpoint = ModelCheckpoint(
    'best_churn_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Callbacks configured!")
print("\nCallback Details:")
print(f"- Early Stopping: Monitoring 'val_loss' with patience=15")
print(f"- Model Checkpoint: Saving best model based on 'val_accuracy'")

Callbacks configured!

Callback Details:
- Early Stopping: Monitoring 'val_loss' with patience=15
- Model Checkpoint: Saving best model based on 'val_accuracy'


## Step 7: Train the Model

Now let's train the model with our prepared data.

In [ ]:
print("Starting model training...\n")
print("=" * 70)

# Train the model
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("\n" + "=" * 70)
print("Model training completed!")

Starting model training...

Epoch 1/100
20/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5213 - loss: 0.6968 
Epoch 1: val_accuracy improved from -inf to 0.67188, saving model to best_churn_model.h5


40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5636 - loss: 0.6831 - val_accuracy: 0.6719 - val_loss: 0.6440
Epoch 2/100
22/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6612 - loss: 0.6502 
Epoch 2: val_accuracy did not improve from 0.67188
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6740 - loss: 0.6393 - val_accuracy: 0.6719 - val_loss: 0.6336
Epoch 3/100
20/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6722 - loss: 0.6318 
Epoch 3: val_accuracy did not improve from 0.67188
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6833 - loss: 0.6207 - val_accuracy: 0.6719 - val_loss: 0.6266
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6771 - loss: 0.6221
Epoch 4: val_accuracy did not improve from 0.67188
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6772 - loss: 0.6218 - val_accuracy: 0.6719 - val_loss: 0.6203
Epoch 5/100
23/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6763 - loss: 0.6183 
Epoch 5: val_accuracy did not improve from 

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6981 - loss: 0.5906 - val_accuracy: 0.6781 - val_loss: 0.6145
Epoch 8/100
20/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6796 - loss: 0.6228 
Epoch 8: val_accuracy did not improve from 0.67813
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6930 - loss: 0.6033 - val_accuracy: 0.6750 - val_loss: 0.6133
Epoch 9/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6959 - loss: 0.5791
Epoch 9: val_accuracy did not improve from 0.67813
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6960 - loss: 0.5790 - val_accuracy: 0.6719 - val_loss: 0.6134
Epoch 10/100
36/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6923 - loss: 0.5788
Epoch 10: val_accuracy did not improve from 0.67813
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6933 - loss: 0.5784 - val_accuracy: 0.6750 - val_loss: 0.6128
Epoch 11/100
26/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7070 - loss: 0.5780
Epoch 11: val_accuracy did not improve from

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7126 - loss: 0.5663 - val_accuracy: 0.6812 - val_loss: 0.6137
Epoch 13/100
27/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7232 - loss: 0.5784
Epoch 13: val_accuracy improved from 0.68125 to 0.68750, saving model to best_churn_model.h5


40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7253 - loss: 0.5735 - val_accuracy: 0.6875 - val_loss: 0.6105
Epoch 14/100
26/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7116 - loss: 0.5680
Epoch 14: val_accuracy did not improve from 0.68750
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7130 - loss: 0.5666 - val_accuracy: 0.6875 - val_loss: 0.6112
Epoch 15/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7218 - loss: 0.5684
Epoch 15: val_accuracy improved from 0.68750 to 0.69063, saving model to best_churn_model.h5


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7217 - loss: 0.5681 - val_accuracy: 0.6906 - val_loss: 0.6123
Epoch 16/100
26/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7439 - loss: 0.5449
Epoch 16: val_accuracy did not improve from 0.69063
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7359 - loss: 0.5486 - val_accuracy: 0.6906 - val_loss: 0.6090
Epoch 17/100
23/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7144 - loss: 0.5820 
Epoch 17: val_accuracy did not improve from 0.69063
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7162 - loss: 0.5721 - val_accuracy: 0.6812 - val_loss: 0.6092
Epoch 18/100
23/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7307 - loss: 0.5656 
Epoch 18: val_accuracy did not improve from 0.69063
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7313 - loss: 0.5593 - val_accuracy: 0.6844 - val_loss: 0.6134
Epoch 19/100
23/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7282 - loss: 0.5526 
Epoch 19: val_accuracy did not improv

## Step 8: Training History Summary

Let's review the training performance.

In [ ]:
# Print final training metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print("\nFinal Training Metrics:")
print("=" * 50)
print(f"Training Accuracy: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")
print(f"Validation Accuracy: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"Training Loss: {final_train_loss:.4f}")
print(f"Validation Loss: {final_val_loss:.4f}")

# Check for overfitting
print("\nOverfitting Analysis:")
if abs(final_train_acc - final_val_acc) < 0.05:
    print("Model shows good generalization (no significant overfitting)")
elif abs(final_train_acc - final_val_acc) < 0.10:
    print("Minor overfitting detected")
else:
    print("Significant overfitting detected")

print(f"Accuracy difference: {abs(final_train_acc - final_val_acc):.4f}")


Final Training Metrics:
Training Accuracy: 0.7578 (75.78%)
Validation Accuracy: 0.6750 (67.50%)
Training Loss: 0.5190
Validation Loss: 0.6064

Overfitting Analysis:
Minor overfitting detected
Accuracy difference: 0.0828


## Step 9: Model Evaluation on Test Set

### 9.1 Make Predictions

In [ ]:
print("Making predictions on test set...\n")

# Get probability predictions
y_pred_prob = model.predict(X_test_scaled, verbose=0)

# Convert probabilities to binary predictions (threshold = 0.5)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Predictions completed!")
print(f"\nPrediction statistics:")
print(f"Predicted Churned: {y_pred.sum()} ({y_pred.sum()/len(y_pred)*100:.2f}%)")
print(f"Predicted Not Churned: {(1-y_pred).sum()} ({(1-y_pred).sum()/len(y_pred)*100:.2f}%)")
print(f"\nActual statistics:")
print(f"Actual Churned: {y_test.sum()} ({y_test.sum()/len(y_test)*100:.2f}%)")
print(f"Actual Not Churned: {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.2f}%)")

Making predictions on test set...

Predictions completed!

Prediction statistics:
Predicted Churned: 62 (15.50%)
Predicted Not Churned: 338 (84.50%)

Actual statistics:
Actual Churned: 123 (30.75%)
Actual Not Churned: 277 (69.25%)


### 9.2 Calculate Performance Metrics

In [ ]:
# Calculate test accuracy
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(f"\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Loss: {test_loss:.4f}")

# Calculate additional metrics
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAdditional Metrics:")
print(f"Accuracy Score: {accuracy:.4f}")


MODEL EVALUATION RESULTS

Test Accuracy: 0.7275 (72.75%)
Test Loss: 0.5626

Additional Metrics:
Accuracy Score: 0.7275
